# Part 1 — Scope, Ownership, and Inputs

### Objective

Use validation data to select the operational model, horizon, stable monitoring window, threshold, and alert policy; then evaluate the fully locked system once on test patients.

### Inputs

Versioned snapshot data, fixed patient splits, locked candidate pipelines, hourly predictions, event-driven replay predictions, and trajectory outputs when available.


# Part 2 — Validate Evaluation Artifacts

### Objective

Confirm that all evaluation inputs are complete and compatible.

### Checks

- Unique model, horizon, and snapshot keys
- Probability bounds
- Required schemas and versions
- Missing predictions
- Patient split disjointness
- Chronological ordering
- Matching label and model horizon definitions

Stop evaluation when artifact integrity fails.


# Part 3 — Run Leakage and Outcome Tests

### Objective

Audit temporal validity before reporting performance.

### Required tests

- Every feature event occurs at or before prediction time.
- Every positive label occurs inside its horizon.
- Snapshots at or after AKI onset are excluded.
- Post-snapshot urine is absent from predictors.
- Preprocessing, weighting, selection, and calibration did not use test information.
- No patient appears in multiple splits.


# Part 4 — Compare Models and Horizons on Validation

### Objective

Compare LR, RF, and boosted trees at 6h, 12h, 24h, and 48h using validation data.

### Metrics

AUROC, AUPRC, sensitivity, specificity, F1, Brier score, and calibration, overall and for each 6h, 12h, 24h, and 48h horizon.

### Rule

Validation results guide selection and are not the final unbiased performance claim.


# Part 5 — Inspect Calibration and Predictor Stability

### Objective

Check whether risk estimates and predictors remain stable across ICU time and horizons.

### Analyses

- Calibration by hours_since_icu and horizon
- Coefficient, importance, or SHAP stability
- Feature-distribution and missingness shifts
- Risk-score volatility between updates
- Monotonic consistency across 6h, 12h, 24h, and 48h risk

Interpret predictors as associations, not causes.


# Part 6 — Select the Stable Monitoring Window

### Objective

Choose the ICU period in which prediction is accurate, calibrated, early enough, and operationally useful.

### Candidate segments

Use a limited prespecified grid such as 8–12, 12–18, 18–24, 24–36, and 36–48 hours.

### Criteria

Discrimination, calibration, sample size, prevalence, risk stability, potential lead time, and false-alert burden.

Lock the monitoring start and end times before test evaluation.


# Part 7 — Select the Operational Horizon and Threshold

### Objective

Choose whether the operational alert should use 6h, 12h, 24h, 48h, or a prespecified combination, then select its risk threshold.

### Guidance

- 48h supports early screening but may create more uncertainty.
- 24h is a candidate primary operational warning horizon.
- 6h and 12h represent higher short-term urgency.

### Rule

Evaluate a limited threshold grid on saved validation predictions; threshold testing never retrains the model.


# Part 8 — Select the Alert Policy on Validation

### Objective

Choose the complete warning logic.

### Candidate components

- Threshold crossing
- Sustained high risk
- Rapid risk increase
- Multi-horizon escalation
- Early-stage stricter confidence requirement
- Alert cooldown and repeat-alert suppression

Select every policy parameter on validation data and define exactly when alerts start, end, and repeat.


# Part 9 — Lock the Evaluation Protocol

### Objective

Freeze the model, horizon, calibration, monitoring window, threshold, alert logic, subgroup definitions, sensitivity analyses, and metrics.

### Rule

After locking, test results must not be used to revise the system. Any revision begins a new labeled development cycle and requires untouched evaluation data.


# Part 10 — Perform Final Snapshot-Level Test Evaluation

### Objective

Evaluate the frozen system on test snapshots.

### Outputs

- AUROC and AUPRC
- Sensitivity, specificity, and F1
- Brier score and calibration plots
- Performance by ICU time and horizon
- Results for 6h, 12h, 24h, and 48h horizons
- Patient-clustered uncertainty intervals

Do not treat correlated hourly rows as independent observations when estimating uncertainty.


# Part 11 — Perform Patient-Level and AKI-Event-Level Evaluation

### Objective

Measure whether the warning system helps at the level that matters operationally.

### Metrics

- Proportion of future AKI events detected
- First-warning lead time per event
- Patients alerted without future AKI
- False alerts per patient-day
- Repeat-alert burden
- Time from eligibility to first alert

Define how multiple alerts map to one AKI event before computing results.


# Part 12 — Evaluate Event-Driven Replay

### Objective

Simulate dashboard operation by recomputing risk whenever a relevant EHR variable changes.

### Checks

- Compare event-driven and hourly risk distributions.
- Measure latency, duplicate predictions, and alert burden.
- Apply an operational debounce or cooldown without hiding clinically meaningful changes.
- Confirm that arbitrary event times do not create material training-deployment mismatch.


# Part 13 — Evaluate the Trajectory Extension

### Objective

Assess future creatinine and urine-output predictions at 6, 12, 24, and 48 hours.

### Metrics

MAE or RMSE, calibration or interval coverage, trajectory shape error, and clinically relevant threshold-crossing performance.

### Stratification

Report accuracy by hours_since_icu, history length, missingness, AKI stage, and ICU type. Early predictions are allowed to be less accurate but must be quantified rather than omitted.


# Part 14 — Compute Interpretation Outputs

### Objective

Explain global model behavior and individual current-risk predictions.

### Outputs

Global feature importance, LR coefficients where applicable, SHAP summaries, patient-level top drivers, feature values, and data recency.

### Rules

Keep patient-level explanations local, verify the explained probability scale, and avoid causal claims.


# Part 15 — Perform Subgroup, System-Era, and Sensitivity Analyses

### Objective

Test whether conclusions depend on population, source system, or study definitions.

### Analyses

- Age, sex, and ICU type
- CareVue versus MetaVision robustness
- Alternative baseline-creatinine definitions
- Creatinine-only versus creatinine plus urine-output KDIGO
- Feature-window and missingness strategies
- Snapshot weighting and alert-policy variants

### Temporal validation note

Do not order patients by shifted MIMIC-III calendar year. Use valid system-era provenance or a later approved external database for true temporal validation.


# Part 16 — Export Dashboard Data and Final Report

### Objective

Create local event-driven dashboard inputs and aggregate research outputs.

### Dashboard views

Current multi-horizon AKI risk, trajectory, warning status, top SHAP drivers, latest measurements, data recency, and patient timeline.

### Final report

Record locked definitions, model and policy versions, snapshot-, patient-, and event-level results, calibration, robustness, limitations, configuration, and Git commit.

Patient-level MIMIC-derived data remain local; shareable demonstrations must use synthetic data.
